# PhenoSuite Quickstart

End-to-end walkthrough that exercises every public surface of the package:

1. Load a plant descriptor XML.
2. Generate the 3D mesh via the C++ engine and export OBJ.
3. Extract phenotyping traits with the pure-Python pipeline.
4. Plot the trait distribution and a 3D view of the leaf splines.

Install the optional extras first:

```bash
pip install -e ".[examples]"
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import phenosuite
from phenosuite import Maize, compute_traits_from_descriptor, from_xml_to_obj

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
DESCRIPTOR = REPO_ROOT / 'plants' / 'plant_0.xml'
OUT_DIR = REPO_ROOT / 'examples' / 'output'
OUT_DIR.mkdir(exist_ok=True)

print(f'phenosuite {phenosuite.__version__}')
print(f'Loading descriptor: {DESCRIPTOR}')
assert DESCRIPTOR.exists(), DESCRIPTOR

## 1. Inspect the descriptor with the pure-Python parser

`compute_traits_from_descriptor` does not require the C++ shared library — it parses the XML and reconstructs leaf splines analytically. Useful in CI and on platforms without a build toolchain.

In [ ]:
traits = compute_traits_from_descriptor(DESCRIPTOR)
print(f'Parsed {len(traits)} leaves\n')
for t in traits:
    angle = t['leaf_angle']
    tip = t['tip_position']
    print(
        f"Leaf {t['leaf_index']:02d} | length={t['leaf_length']:.3f} m | "
        f"azimuth={angle['azimuth_deg']:6.2f}° | inclination={angle['inclination_deg']:5.2f}° | "
        f"tip=({tip['x']:+.3f}, {tip['y']:+.3f}, {tip['z']:+.3f})"
    )

## 2. Generate the OBJ mesh via the C++ engine

`from_xml_to_obj` is a one-shot helper that loads the descriptor, rebuilds the mesh, and writes Wavefront OBJ + MTL files. It requires the compiled shared library (bundled for Windows, built from source on Linux/macOS — see the README).

In [ ]:
obj_path = OUT_DIR / 'plant_0.obj'
ok = from_xml_to_obj(
    str(DESCRIPTOR), str(obj_path),
    leaf_texture_path=str(REPO_ROOT / 'plants' / 'maize_leaf.png'),
    stem_texture_path=str(REPO_ROOT / 'plants' / 'maize_stem_texture.png'),
)
assert ok, 'Failed to export OBJ'
print(f'Wrote {obj_path} ({obj_path.stat().st_size / 1024:.1f} KB)')

## 3. Compare the two pipelines on the same plant

Both pipelines return the leaf-stem connection point. The Python pipeline reconstructs splines analytically; the C++ engine reports the actual spline used by the renderer. They should agree to within a few millimeters.

In [ ]:
with Maize() as gen:
    gen.load_xml(str(DESCRIPTOR))
    cpp_traits = [gen.get_leaf_spline_traits(i) for i in range(gen.leaf_spline_trait_count())]
    print(f'C++ engine reports {gen.triangle_count()} triangles for {len(cpp_traits)} leaves\n')

diffs = []
for py_t, cpp_t in zip(traits, cpp_traits):
    py_cp = py_t['connection_point']
    cpp_cp = cpp_t['connection_point']
    d = np.sqrt(sum((py_cp[k] - cpp_cp[k]) ** 2 for k in 'xyz'))
    diffs.append(d)

print(f'Connection-point discrepancy (Python vs C++):')
print(f'  max  : {max(diffs) * 1000:.3f} mm')
print(f'  mean : {np.mean(diffs) * 1000:.3f} mm')

## 4. Plot the trait distribution

Per-leaf bar chart of length, azimuth, and inclination.

In [ ]:
leaf_indices = [t['leaf_index'] for t in traits]
lengths = [t['leaf_length'] for t in traits]
azimuths = [t['leaf_angle']['azimuth_deg'] for t in traits]
inclinations = [t['leaf_angle']['inclination_deg'] for t in traits]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].bar(leaf_indices, lengths, color='#3a8c5e')
axes[0].set_xlabel('Leaf index'); axes[0].set_ylabel('Length (m)'); axes[0].set_title('Leaf length')
axes[1].bar(leaf_indices, azimuths, color='#3672b5')
axes[1].set_xlabel('Leaf index'); axes[1].set_ylabel('Azimuth (°)'); axes[1].set_title('Azimuth')
axes[2].bar(leaf_indices, inclinations, color='#c25b56')
axes[2].set_xlabel('Leaf index'); axes[2].set_ylabel('Inclination (°)'); axes[2].set_title('Inclination')
plt.tight_layout(); plt.show()

## 5. 3D view of the reconstructed leaf splines

World-space coordinate frame: **+Y is up** (stem growth direction), X/Z is the ground plane.

In [ ]:
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection='3d')

cmap = plt.get_cmap('viridis')
for t in traits:
    spline = np.array(t['center_spline'])
    color = cmap(t['leaf_index'] / max(1, len(traits) - 1))
    ax.plot(spline[:, 0], spline[:, 2], spline[:, 1], color=color, lw=2)
    cp = t['connection_point']
    tip = t['tip_position']
    ax.scatter([cp['x']], [cp['z']], [cp['y']], color=color, marker='o', s=20)
    ax.scatter([tip['x']], [tip['z']], [tip['y']], color=color, marker='^', s=30)

stem_y = max(t['connection_point']['y'] for t in traits)
ax.plot([0, 0], [0, 0], [0, stem_y], color='black', lw=3, alpha=0.5)

ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)'); ax.set_zlabel('Y, up (m)')
ax.set_title('Reconstructed leaf splines (○ = connection, △ = tip)')
ax.set_box_aspect([1, 1, 1.4])
plt.tight_layout(); plt.show()

## 6. Save traits as XML for downstream analysis

`write_traits_xml` produces a flat `<traits>` document — one `<leaf>` element per record, with leaf angle, length, connection point, and tip position. The schema is intentionally simple and is intended for downstream pipelines, not for round-tripping back into a descriptor.

In [ ]:
from phenosuite import write_traits_xml

out_traits = OUT_DIR / 'traits.xml'
write_traits_xml(out_traits, DESCRIPTOR, traits)
print(f'Wrote {out_traits}')
print()
print(out_traits.read_text(encoding='utf-8')[:600] + '...')